In [1]:
# High-performance VAE imputation with deterministic settings
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy import special
import warnings
import random
import os
import gc
import time
import hashlib

warnings.filterwarnings("ignore")

# Check the TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# ==================== Reproducibility settings ====================
SEED = 42

def set_complete_determinism(seed=42):
    """Configure deterministic execution for reproducibility."""
    # Python random seed
    random.seed(seed)
    
    # Hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy random seed
    np.random.seed(seed)
    
    # TensorFlow random seed
    tf.random.set_seed(seed)
    
    # Enable deterministic TensorFlow operations
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
    # Disable multithreading for determinism
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
    
    # Clear the previous session
    tf.keras.backend.clear_session()
    
    # Set floating-point precision
    tf.keras.backend.set_floatx('float32')

# Initialize deterministic mode
set_complete_determinism(SEED)

# Device settings for CPU reproducibility
def setup_computation_device(use_cpu_only=True):
    """Set the computation device."""
    if use_cpu_only:
        # Use the CPU for reproducibility
        os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
        print("Running on the CPU for reproducibility")
    else:
        # Enable deterministic operations when using the GPU
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            try:
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                print(f"Found {len(gpus)} GPU devices")
            except RuntimeError as e:
                print(f"GPU configuration error: {e}")
        else:
            print("No GPU found. Running on the CPU.")

# Use the CPU for reproducibility
setup_computation_device(use_cpu_only=True)

# Clear the session
gc.collect()

# Implement the inverse Box-Cox transform
def inv_boxcox(y, lambda_param):
    """Apply the inverse Box-Cox transform."""
    if lambda_param == 0:
        return np.exp(y)
    else:
        return np.power(lambda_param * y + 1, 1 / lambda_param)

# 1. Data loading and preprocessing
def load_and_preprocess_data():
    """Load the data and perform initial preprocessing."""
    # Set the seed before loading the data
    np.random.seed(SEED)
    
    data = pd.read_csv("./Rock.csv", low_memory=True)
    print(f"Original data shape: {data.shape}")
    
    # Keep a copy of the original data
    original_data = data.copy()
    
    # Drop nonnumeric columns
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # Drop ID columns
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # Drop the L(m) column
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # Drop columns with more than 30% missing values
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.30].index
    data = data.drop(columns=cols_to_drop)
    
    # Convert data types
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    # Reset the index for consistency
    data = data.reset_index(drop=True)
    
    print(f"Final data shape: {data.shape}")
    print(f"Features: {list(data.columns)}")
    
    return data, original_data

# 2. Outlier treatment and feature transforms
def enhanced_outlier_handling(df):
    """Treat outliers using a separate method for Wsm."""
    # Reset random seeds
    np.random.seed(SEED)
    
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # Calculate descriptive statistics
        q1, q25, q50, q75, q99 = np.percentile(values, [1, 25, 50, 75, 99])
        iqr = q75 - q25
        std_val = np.std(values)
        mean_val = np.mean(values)
        
        # Special handling for V(m³)
        if 'V' in column:
            print(f"{column}: applying a quantile transform to a highly skewed feature")
            
            qt = QuantileTransformer(n_quantiles=min(1000, len(values)), 
                                    output_distribution='normal', 
                                    random_state=SEED,
                                    subsample=100000)  # Fixed subsample size
            values_transformed = qt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {'type': 'quantile', 'transformer': qt}
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            
        elif 'As' in column:
            print(f"{column}: applying the Box-Cox transform")
            
            min_val = values.min()
            if min_val <= 0:
                shift = 1 - min_val
                values = values + shift
            else:
                shift = 0
            
            values_transformed, lambda_param = stats.boxcox(values)
            
            q1, q3 = np.percentile(values_transformed, [25, 75])
            iqr = q3 - q1
            lower = q1 - 3 * iqr
            upper = q3 + 3 * iqr
            values_transformed = np.clip(values_transformed, lower, upper)
            
            transform_info[column] = {'type': 'boxcox', 'lambda': lambda_param, 'shift': shift}
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            
        elif 'H' in column:
            print(f"{column}: applying the Box-Cox transform")
            
            min_val = values.min()
            if min_val <= 0:
                shift = 1 - min_val
                values = values + shift
            else:
                shift = 0
            
            values_transformed, lambda_param = stats.boxcox(values)
            
            q1, q3 = np.percentile(values_transformed, [25, 75])
            iqr = q3 - q1
            lower = q1 - 3 * iqr
            upper = q3 + 3 * iqr
            values_transformed = np.clip(values_transformed, lower, upper)
            
            transform_info[column] = {'type': 'boxcox', 'lambda': lambda_param, 'shift': shift}
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            
        elif 'Wsm' in column:
            print(f"{column}: applying feature-specific preprocessing")
            
            try:
                min_val = values.min()
                if min_val <= 0:
                    shift = 1.1 - min_val
                    values_shifted = values + shift
                else:
                    shift = 0.1
                    values_shifted = values + shift
                
                values_transformed, lambda_param = stats.boxcox(values_shifted)
                
                if np.std(values_transformed) > 0:
                    q1, q3 = np.percentile(values_transformed, [10, 90])
                    iqr = q3 - q1
                    lower = q1 - 2 * iqr
                    upper = q3 + 2 * iqr
                    values_transformed = np.clip(values_transformed, lower, upper)
                    
                    transform_info[column] = {'type': 'boxcox', 'lambda': lambda_param, 'shift': shift}
                    print(f"  {column}: Box-Cox transform succeeded, lambda={lambda_param:.4f}")
                else:
                    raise ValueError("Variance is zero after the Box-Cox transform")
                    
            except:
                print(f"  {column}: Box-Cox failed. Applying a quantile transform.")
                
                q1, q3 = np.percentile(values, [5, 95])
                iqr = q3 - q1
                lower = q1 - 2 * iqr
                upper = q3 + 2 * iqr
                values_clipped = np.clip(values, lower, upper)
                
                qt = QuantileTransformer(n_quantiles=min(500, len(values_clipped)), 
                                        output_distribution='normal', 
                                        random_state=SEED,
                                        subsample=100000)
                values_transformed = qt.fit_transform(values_clipped.reshape(-1, 1)).flatten()
                
                transform_info[column] = {'type': 'quantile', 'transformer': qt}
            
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            
        elif 'θ' in column:
            print(f"{column}: minimal preprocessing to preserve the distribution")
            
            lower, upper = np.percentile(values, [0.5, 99.5])
            values = np.clip(values, lower, upper)
            transform_info[column] = {'type': 'none', 'shift': 0}
            df_clean[column] = df_clean[column].astype(values.dtype, copy=False)
            df_clean.loc[mask, column] = values
            
        else:
            print(f"{column}: applying robust outlier treatment")
            
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 2.5 * iqr
            upper = q3 + 2.5 * iqr
            values = np.clip(values, lower, upper)
            
            transform_info[column] = {'type': 'clip', 'shift': 0}
            df_clean[column] = df_clean[column].astype(values.dtype, copy=False)
            df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. Enhanced scaling
def enhanced_normalization(data):
    """Apply feature-specific scaling."""
    # Reset random seeds
    np.random.seed(SEED)
    
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            if 'V' in column:
                scaler = StandardScaler()
            elif 'Wsm' in column:
                scaler = StandardScaler()
            elif 'H' in column or 'As' in column:
                scaler = RobustScaler(quantile_range=(10, 90))
            else:
                scaler = RobustScaler(quantile_range=(5, 95))
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {'scaler': scaler, 'min': min_val, 'max': max_val}
    
    return data_normalized, scalers

# Inverse scaling
def inverse_transform_array(data_normalized_array, scalers, transform_info, original_columns, original_index=None):
    """Convert the scaled NumPy array to a DataFrame in the original units."""
    n_samples = data_normalized_array.shape[0]
    
    if original_index is None:
        original_index = range(n_samples)
    
    data_original = pd.DataFrame(index=original_index, columns=original_columns)
    
    for i, column in enumerate(original_columns):
        if column not in scalers:
            data_original[column] = data_normalized_array[:, i]
            continue
            
        values = data_normalized_array[:, i].copy()
        
        scaler_info = scalers[column]
        min_val = scaler_info['min']
        max_val = scaler_info['max']
        
        values = (values - 0.1) / 0.8 * (max_val - min_val) + min_val
        values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        if column in transform_info:
            trans_type = transform_info[column]['type']
            
            if trans_type == 'quantile':
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'boxcox':
                lambda_param = transform_info[column]['lambda']
                shift = transform_info[column]['shift']
                values = inv_boxcox(values, lambda_param) - shift
            elif trans_type == 'log':
                shift = transform_info[column]['shift']
                values = np.expm1(values) - shift
            elif trans_type == 'sqrt':
                shift = transform_info[column]['shift']
                values = values ** 2 - shift
        
        data_original[column] = values
    
    return data_original

# 4. Deterministic VAE
class DeterministicVAE:
    def __init__(self, data_dim, latent_dim=10, learning_rate=0.0005, seed=42):
        """Deterministic VAE model."""
        self.data_dim = data_dim
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.seed = seed
        
        # Reset all seeds
        set_complete_determinism(seed)
        
        # Build the model
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # Optimizer
        self.optimizer = optimizers.Adam(learning_rate=learning_rate)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
    
    def _build_encoder(self):
        """Build the deterministic encoder."""
        # Reset the seed
        tf.random.set_seed(self.seed)
        np.random.seed(self.seed)
        
        inputs = layers.Input(shape=(self.data_dim,))
        
        # Use fixed initializer seeds
        initializer = initializers.GlorotUniform(seed=self.seed)
        
        x = layers.Dense(64, kernel_initializer=initializer)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.2, seed=self.seed)(x, training=True)  # Use a fixed dropout seed
        
        x = layers.Dense(32, kernel_initializer=initializer)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.15, seed=self.seed+1)(x, training=True)
        
        x = layers.Dense(16, kernel_initializer=initializer)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        z_mean = layers.Dense(self.latent_dim, kernel_initializer=initializer)(x)
        z_log_var = layers.Dense(self.latent_dim, kernel_initializer=initializer)(x)
        
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -3, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """Build the deterministic decoder."""
        # Reset the seed
        tf.random.set_seed(self.seed)
        np.random.seed(self.seed)
        
        inputs = layers.Input(shape=(self.latent_dim,))
        
        initializer = initializers.GlorotUniform(seed=self.seed)
        
        x = layers.Dense(16, kernel_initializer=initializer)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Dense(32, kernel_initializer=initializer)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.15, seed=self.seed+2)(x, training=True)
        
        x = layers.Dense(64, kernel_initializer=initializer)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.2, seed=self.seed+3)(x, training=True)
        
        outputs = layers.Dense(self.data_dim, activation='sigmoid', kernel_initializer=initializer)(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """Sample with a seeded generator using the reparameterization trick."""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        # Draw random values with a fixed seed
        tf.random.set_seed(self.seed)
        epsilon = tf.random.normal(shape=(batch, dim), seed=self.seed)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.05
    
    @tf.function
    def train_step(self, x_batch, mask_batch, kl_weight=0.0005):
        """Run one training step."""
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encoder(x_batch, training=True)
            z = self._sample(z_mean, z_log_var)
            reconstructed = self.decoder(z, training=True)
            
            mse = tf.square(x_batch - reconstructed) * mask_batch
            reconstruction_loss = tf.reduce_sum(mse) / tf.reduce_sum(mask_batch)
            
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            total_loss = reconstruction_loss + kl_weight * kl_loss
        
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss
    
    def fit(self, data, mask, validation_split=0.2, epochs=500, batch_size=64):
        """Train the model using deterministic settings."""
        n_samples = len(data)
        indices = np.arange(n_samples)
        
        # Split the data with a fixed seed
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train = data[train_indices]
        x_val = data[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"Training set: {len(x_train)} samples")
        print(f"Validation set: {len(x_val)} samples")
        
        patience = 50
        patience_counter = 0
        
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            self.learning_rate,
            decay_steps=1000,
            decay_rate=0.96,
            staircase=True
        )
        self.optimizer.learning_rate = lr_schedule
        
        for epoch in range(epochs):
            kl_weight = min(0.001, 0.0001 * (epoch / 100))
            
            # Shuffle with a fixed seed
            np.random.seed(self.seed + epoch)
            train_indices_shuffled = np.random.permutation(len(x_train))
            train_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss = self.train_step(
                    x_train[batch_indices],
                    mask_train[batch_indices],
                    kl_weight
                )
                train_loss += loss
                n_batches += 1
            
            train_loss /= n_batches
            
            if epoch % 10 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val), batch_size):
                    z_mean, z_log_var = self.encoder(x_val[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 10
                
                if epoch % 50 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data, mask):
        """Impute missing values using deterministic sampling."""
        imputed = np.zeros_like(data)
        batch_size = 128
        
        # Fixed seed
        tf.random.set_seed(self.seed)
        
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            predictions = []
            for j in range(20):
                # Use a different fixed seed for each draw
                tf.random.set_seed(self.seed + j)
                z_mean, z_log_var = self.encoder(batch_data, training=False)
                z = self._sample(z_mean, z_log_var)
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            pred_mean = np.mean(predictions, axis=0)
            imputed[i:i+batch_size] = batch_data * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 5. Deterministic deep learning models
class DeterministicDeepModels:
    def __init__(self, seed=42):
        self.seed = seed
        self.models = {}
        self.scalers = {}
    
    def _build_feature_model(self, input_dim, feature_name=None):
        """Build a deterministic deep learning model for one feature."""
        # Reset the seed
        tf.random.set_seed(self.seed)
        np.random.seed(self.seed)
        
        initializer = initializers.GlorotUniform(seed=self.seed)
        
        if feature_name and 'Wsm' in feature_name:
            model = keras.Sequential([
                layers.Dense(64, activation='relu', input_shape=(input_dim,), 
                           kernel_initializer=initializer),
                layers.BatchNormalization(),
                layers.Dropout(0.2, seed=self.seed),
                layers.Dense(32, activation='relu', kernel_initializer=initializer),
                layers.BatchNormalization(),
                layers.Dropout(0.15, seed=self.seed+1),
                layers.Dense(16, activation='relu', kernel_initializer=initializer),
                layers.Dense(8, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        else:
            model = keras.Sequential([
                layers.Dense(32, activation='relu', input_shape=(input_dim,), 
                           kernel_initializer=initializer),
                layers.Dropout(0.2, seed=self.seed),
                layers.Dense(16, activation='relu', kernel_initializer=initializer),
                layers.Dropout(0.1, seed=self.seed+1),
                layers.Dense(8, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        
        model.compile(optimizer='adam', loss='mse')
        return model
    
    def fit(self, data, mask, column_names=None):
        """Train a deterministic deep learning model for each feature."""
        print("Training feature-specific deep learning models...")
        
        for col_idx in range(data.shape[1]):
            col_name = column_names[col_idx] if column_names is not None else f"feature_{col_idx}"
            obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 100:
                continue
            
            # Fixed seed
            np.random.seed(self.seed + col_idx)
            
            X_train = np.delete(data[obs_mask], col_idx, axis=1)
            y_train = data[obs_mask, col_idx]
            
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            X_train_scaled = scaler_x.fit_transform(X_train)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            
            model = self._build_feature_model(X_train.shape[1], col_name)
            
            patience = 20 if 'Wsm' in col_name else 10
            early_stop = callbacks.EarlyStopping(monitor='loss', patience=patience, restore_best_weights=True)
            
            epochs = 200 if 'Wsm' in col_name else 100
            
            model.fit(
                X_train_scaled, y_train_scaled,
                epochs=epochs,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop]
            )
            
            self.models[col_idx] = model
            self.scalers[col_idx] = {'x': scaler_x, 'y': scaler_y}
    
    def impute(self, data, mask):
        """Impute missing values."""
        imputed = data.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(imputed[miss_mask], col_idx, axis=1)
            X_miss_scaled = self.scalers[col_idx]['x'].transform(X_miss)
            pred_scaled = model.predict(X_miss_scaled, verbose=0).flatten()
            pred = self.scalers[col_idx]['y'].inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
            
            imputed[miss_mask, col_idx] = pred
        
        return imputed

# 6. Deterministic ensemble
class DeterministicEnsemble:
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = []
        self.feature_weights = {}
    
    def fit(self, data, mask, column_names):
        """Train the deterministic ensemble."""
        print("\nTraining the deterministic ensemble...")
        
        # Reset the seed
        np.random.seed(self.seed)
        
        # 1. KNN
        print("1. KNN imputer...")
        knn = KNNImputer(n_neighbors=15, weights='distance')
        knn.fit(data)
        self.models.append(('knn', knn))
        
        # 2. GradientBoosting IterativeImputer
        print("2. GradientBoosting IterativeImputer...")
        gb_mice = IterativeImputer(
            estimator=GradientBoostingRegressor(
                n_estimators=150,
                max_depth=5,
                learning_rate=0.1,
                subsample=0.8,
                random_state=self.seed
            ),
            max_iter=25,
            random_state=self.seed
        )
        gb_mice.fit(data)
        self.models.append(('gb_mice', gb_mice))
        
        # 3. IterativeImputer with a random forest
        print("3. IterativeImputer with a random forest...")
        rf_mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=150,
                max_depth=8,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        rf_mice.fit(data)
        self.models.append(('rf_mice', rf_mice))
        
        # 4. ExtraTrees IterativeImputer
        print("4. ExtraTrees IterativeImputer...")
        et_mice = IterativeImputer(
            estimator=ExtraTreesRegressor(
                n_estimators=150,
                max_depth=8,
                min_samples_split=3,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        et_mice.fit(data)
        self.models.append(('et_mice', et_mice))
        
        # 5. Deterministic deep learning models
        print("5. Deterministic deep learning models...")
        fs_model = DeterministicDeepModels(seed=self.seed)
        fs_model.fit(data, mask, column_names)
        self.models.append(('feature_deep', fs_model))
        
        # 6. Deterministic VAE
        print("6. Deterministic VAE model...")
        vae = DeterministicVAE(data_dim=self.data_dim, latent_dim=10, learning_rate=0.0005, seed=self.seed)
        
        mean_imputer = SimpleImputer(strategy='mean')
        mean_imputer.fit(data)
        initial_imputed = mean_imputer.transform(data)
        data_filled = np.where(mask == 1, data, initial_imputed)
        
        vae.fit(data_filled, mask, epochs=500, batch_size=64)
        self.models.append(('vae', vae))
        
        # Set the weights for each feature
        for i, col_name in enumerate(column_names):
            if 'V' in col_name:
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.25,
                    'rf_mice': 0.25,
                    'et_mice': 0.20,
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
            elif 'As' in col_name:
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.20,
                    'rf_mice': 0.25,
                    'et_mice': 0.20,
                    'feature_deep': 0.15,
                    'vae': 0.05
                }
            elif 'H' in col_name:
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.20,
                    'rf_mice': 0.20,
                    'et_mice': 0.15,
                    'feature_deep': 0.20,
                    'vae': 0.10
                }
            elif 'Wsm' in col_name:
                self.feature_weights[i] = {
                    'knn': 0.05,
                    'gb_mice': 0.35,
                    'rf_mice': 0.30,
                    'et_mice': 0.20,
                    'feature_deep': 0.08,
                    'vae': 0.02
                }
            else:
                self.feature_weights[i] = {
                    'knn': 0.20,
                    'gb_mice': 0.20,
                    'rf_mice': 0.20,
                    'et_mice': 0.15,
                    'feature_deep': 0.15,
                    'vae': 0.10
                }
        
        print("Ensemble training complete")
    
    def impute(self, data, mask):
        """Predict using the deterministic ensemble."""
        n_samples, n_features = data.shape
        imputed = np.zeros_like(data)
        
        all_predictions = {}
        for name, model in self.models:
            if name == 'vae':
                mean_imputer = SimpleImputer(strategy='mean')
                mean_imputer.fit(data)
                initial_imputed = mean_imputer.transform(data)
                data_filled = np.where(mask == 1, data, initial_imputed)
                pred = model.impute(data_filled, mask)
            elif name == 'feature_deep':
                pred = model.impute(data, mask)
            else:
                pred = model.transform(data)
            all_predictions[name] = pred
        
        for feature_idx in range(n_features):
            if feature_idx in self.feature_weights:
                weights = self.feature_weights[feature_idx]
                feature_pred = np.zeros(n_samples)
                
                for model_name, weight in weights.items():
                    if model_name in all_predictions:
                        feature_pred += all_predictions[model_name][:, feature_idx] * weight
                
                imputed[:, feature_idx] = feature_pred
            else:
                feature_pred = np.zeros(n_samples)
                n_models = len(all_predictions)
                for pred in all_predictions.values():
                    feature_pred += pred[:, feature_idx] / n_models
                imputed[:, feature_idx] = feature_pred
        
        imputed = data * mask + imputed * (1 - mask)
        
        return imputed

# 7. Deterministic evaluation
def deterministic_evaluate(model, data_array, mask, column_names, 
                         data_raw, scalers, transform_info):
    """Evaluate model performance using deterministic cross-validation."""
    print("\n=== Deterministic model evaluation ===")
    
    results = []
    all_observations = []
    
    original_index = data_raw.index
    
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        # Use a fixed seed for KFold
        kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
        
        all_true = []
        all_pred = []
        all_indices = []
        
        for train_idx, test_idx in kf.split(obs_indices):
            test_indices = obs_indices[test_idx]
            
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            imputed_normalized = model.impute(data_array, test_mask)
            
            imputed_original = inverse_transform_array(
                imputed_normalized, scalers, transform_info, 
                column_names, original_index
            )
            
            true_values_original = data_raw.iloc[test_indices][col_name].values
            pred_values_original = imputed_original[col_name].iloc[test_indices].values
            
            all_true.extend(true_values_original)
            all_pred.extend(pred_values_original)
            all_indices.extend(test_indices)
            
            for idx, true_val, pred_val in zip(test_indices, true_values_original, pred_values_original):
                all_observations.append({
                    'Row Index': idx,
                    'Feature': col_name,
                    'Observed Value': true_val,
                    'Predicted Value': pred_val,
                    'Absolute Error': abs(true_val - pred_val)
                })
        
        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        mae = mean_absolute_error(all_true, all_pred)
        rmse = np.sqrt(mean_squared_error(all_true, all_pred))
        
        ss_res = np.sum((all_true - all_pred) ** 2)
        ss_tot = np.sum((all_true - np.mean(all_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        if np.std(all_true) > 0 and np.std(all_pred) > 0:
            correlation = np.corrcoef(all_true, all_pred)[0, 1]
        else:
            correlation = 0
        
        results.append({
            'Feature': col_name,
            'R² (Original Scale)': r2,
            'Correlation': correlation,
            'MAE': mae,
            'RMSE': rmse,
            'Sample Count': len(all_true)
        })
        
        print(f"{col_name}: R²={r2:.4f}, correlation={correlation:.4f}, MAE={mae:.4f}")
    
    return pd.DataFrame(results), pd.DataFrame(all_observations)

# 8. Main function
def main():
    """Run the workflow with deterministic settings."""
    print("=" * 80)
    print("VAE imputation with deterministic settings")
    print("Running on the CPU for reproducibility")
    print("=" * 80)
    
    try:
        # Deterministic settings
        set_complete_determinism(SEED)
        
        # 1. Load the data
        print("\n1. Loading data...")
        data_raw, original_data = load_and_preprocess_data()
        data_raw_copy = data_raw.copy()
        
        # 2. Treat outliers
        print("\n2. Treating outliers...")
        data_clean, transform_info = enhanced_outlier_handling(data_raw)
        
        # 3. Scale the data
        print("\n3. Scaling the data...")
        data_normalized, scalers = enhanced_normalization(data_clean)
        
        # Create the mask
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        
        print(f"\nData shape: {data_array.shape}")
        print(f"Missing rate: {(1 - mask.mean())*100:.2f}%")
        
        # 4. Train the deterministic ensemble
        print("\n4. Training the deterministic ensemble...")
        model = DeterministicEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_normalized.columns)
        
        # 5. Deterministic evaluation
        print("\n5. Running deterministic evaluation...")
        eval_results, observations_df = deterministic_evaluate(
            model, data_array, mask, data_normalized.columns, 
            data_raw_copy, scalers, transform_info
        )
        
        # 6. Final imputation
        print("\n6. Generating the final imputed data...")
        final_imputed_normalized = model.impute(data_array, mask)
        
        final_imputed_original = inverse_transform_array(
            final_imputed_normalized, scalers, transform_info, 
            data_normalized.columns, data_raw_copy.index
        )
        
        final_result = data_raw_copy.astype(np.float64)
        for col in final_result.columns:
            missing_mask = data_raw_copy[col].isna()
            final_result.loc[missing_mask, col] = final_imputed_original.loc[missing_mask, col]
        
        # 7. Report the results
        print("\n" + "=" * 80)
        print("Model performance summary (deterministic run)")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['Feature']}:")
            print(f"  R²: {row['R² (Original Scale)']:.4f}")
            print(f"  Correlation: {row['Correlation']:.4f}")
            print(f"  MAE: {row['MAE']:.4f}")
            print(f"  RMSE: {row['RMSE']:.4f}")
        
        print(f"\nMean R²: {eval_results['R² (Original Scale)'].mean():.4f}")
        print(f"Minimum R²: {eval_results['R² (Original Scale)'].min():.4f}")
        
        # Hash the results to check reproducibility
        result_hash = hashlib.md5(str(eval_results['R² (Original Scale)'].values).encode()).hexdigest()
        print(f"\nResult hash for reproducibility checks: {result_hash}")
        
        # Save the results
        observations_df.to_csv("observed_vs_predicted_rock.csv", index=False, encoding='utf-8-sig')
        final_result.to_csv("imputed_rock.csv", index=False, encoding='utf-8-sig')
        eval_results.to_csv("evaluation_rock.csv", index=False, encoding='utf-8-sig')
        
        # Save the hash for reproducibility checks
        with open("result_hash_rock.txt", "w", encoding="utf-8") as f:
            f.write(f"R² result hash: {result_hash}\n")
            f.write(f"Random seed: {SEED}\n")
            f.write(f"TensorFlow version: {tf.__version__}\n")
        
        print("\nFiles saved:")
        print("1. observed_vs_predicted_rock.csv")
        print("2. imputed_rock.csv")
        print("3. evaluation_rock.csv")
        print("4. result_hash_rock.txt")
        
        print("\n" + "=" * 80)
        print("🔒 Deterministic execution completed")
        print("Compare result hashes across runs to check reproducibility.")
        print("=" * 80)
        
        return final_result, eval_results, observations_df
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

if __name__ == "__main__":
    print("\nStarting the workflow...")
    print("Reproducibility settings:")
    print("1. CPU execution")
    print("2. Single-threaded execution")
    print("3. Fixed random seeds")
    print("4. Deterministic operations")
    print("5. Fixed initializer seeds")
    
    start_time = time.time()
    result, evaluation, observations = main()
    
    if result is not None:
        print(f"\n✅ Run complete!")
        print(f"Elapsed time: {(time.time() - start_time)/60:.2f} minutes")
        print("\nRerun the workflow to check reproducibility.")
    else:
        print("\n❌ Run failed")
    
    gc.collect()
    

TensorFlow version: 2.21.0

Running on the CPU for reproducibility

Starting the workflow...
Reproducibility settings:
1. CPU execution
2. Single-threaded execution
3. Fixed random seeds
4. Deterministic operations
5. Fixed initializer seeds
VAE imputation with deterministic settings
Running on the CPU for reproducibility

1. Loading data...

Error: [Errno 2] No such file or directory: './Rock.csv'

❌ Run failed


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10192\2690939365.py", line 977, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10192\2690939365.py", line 102, in load_and_preprocess_data
    data = pd.read_csv("./Rock.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 1645, in

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    Add the L(m) column to the imputed results.
    """
    # 1. Read the source data to retrieve L(m)
    original_data = pd.read_csv("./Rock.csv", low_memory=True)
    
    # 2. Check whether L(m) is present
    if 'L(m)' not in original_data.columns:
        print("The source data does not contain L(m)")
        return
    
    # 3. Extract L(m)
    L_column = original_data['L(m)'].copy()
    print(f"Extracted L(m) with {len(L_column)} rows")
    
    # 4. Read the imputed results
    try:
        imputed_results = pd.read_csv("imputed_rock.csv")
        print(f"Imputed data shape: {imputed_results.shape}")
    except FileNotFoundError:
        print("Imputed results file not found. Run the imputation cell first.")
        return
    
    # 5. Check that the row counts match
    if len(imputed_results) != len(L_column):
        print(f"Warning: imputed results have {len(imputed_results)} rows, while L(m) has {len(L_column)} rows")
        # If row counts differ, align both tables to the shorter length
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. Add L(m) to the results
    imputed_results['L(m)'] = L_column.values
    
    # 7. Save the updated results
    output_filename = "imputed_rock_with_runout.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\nAdded L(m) to the imputed results.")
    print(f"Saved the new file as: {output_filename}")
    print(f"Final data shape: {imputed_results.shape}")
    print(f"Column names: {list(imputed_results.columns)}")
    
    # 8. Report summary statistics for L(m)
    print("\nL(m) summary statistics:")
    print(f"  Data type: {L_column.dtype}")
    print(f"  Unique values: {L_column.nunique()}")
    print(f"  Missing values: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  Minimum: {L_column.min()}")
        print(f"  Maximum: {L_column.max()}")
        print(f"  Mean: {L_column.mean():.2f}")
    else:
        print(f"  First five values: {L_column.head().tolist()}")
    
    return imputed_results

# Run the function
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

Extracted L(m) with 17823 rows
Imputed data shape: (17823, 5)

Added L(m) to the imputed results.
Saved the new file as: imputed_rock_with_runout.csv
Final data shape: (17823, 6)
Column names: ['H(m)', 'As(m²）', 'V(m³)', 'Wsm(m)', 'θ(°)', 'L(m)']

L(m) summary statistics:
  Data type: float64
  Unique values: 8055
  Missing values: 0
  Minimum: 0.263488676
  Maximum: 15600.0
  Mean: 95.70
